In [ ]:
# good thing for us, we do not have to implement the multiprocessing part or the loading part
# all these things are already implemented in the DataLoader class
# and pytorch. So lets use that.

import torch
import glob
from torch.utils.data import DataLoader, IterableDataset, get_worker_info
import math
import os


class CustomDataset(IterableDataset):

    def __init__(self, path, cs):
        self.chunk_size = cs
        self.path = path
        self.all_files = glob.glob(os.path.join(self.path, "*.txt"))
        if not self.all_files:
            raise ValueError(f"No .txt files found under {path!r}")

    def _iter_chunks_for_line(self, line: str):
        cs = self.chunk_size
        for i in range(0, len(line), cs):
            chunk = line[i:i+cs]
            yield chunk

    def __iter__(self):
        worker_info = get_worker_info()
        if worker_info is None:
            files = self.all_files
        else:
            num_workers = worker_info.num_workers
            worker_id = worker_info.id
            per = int(math.ceil(len(self.all_files) / num_workers))
            start_idx = worker_id * per
            end_idx = min(start_idx + per, len(self.all_files))
            files = self.all_files[start_idx:end_idx]

        while True:
            for path in files:
                with open(path, "r", encoding=self.encoding, errors="replace") as f:
                    for line in f:
                        yield from self._iter_chunks_for_line(line)


/Users/devansh20la/Documents/interview-prep/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:20: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  device: torch.device = torch.device(torch._C._get_default_device()),  # torch.device('cpu'),
